In [ ]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class PrepareBaseModelConfig:
    root_dir: Path
    base_model_path: Path
    updated_base_model_path: Path
    params_image_size: list
    params_include_top: bool
    params_weights: str
    params_learning_rate: float
    params_classes: int

In [43]:
from cnnClassifier.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH
from cnnClassifier.utils.common import read_yaml, create_directories

In [ ]:
class ConfiguarationManager:
    def __init__(self,
                 config_file_path = CONFIG_FILE_PATH,
                 params_file_path = PARAMS_FILE_PATH):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        create_directories([self.config.artifacts_root])

    def get_prepare_base_model_config(self) -> PrepareBaseModelConfig:
        config = self.config.prepare_base_model

        create_directories([config.root_dir])

        prepare_base_model_config = PrepareBaseModelConfig(
            root_dir=Path(config.root_dir),
            base_model_path=Path(config.base_model_path),
            updated_base_model_path=Path(config.updated_base_model_path),
            params_image_size=self.params.IMAGE_SIZE,
            params_include_top=self.params.INCLUDE_TOP,
            params_weights=self.params.WEIGHTS,
            params_classes=self.params.CLASSES, 
            params_learning_rate=self.params.LEARNING_RATE
        )

        return prepare_base_model_config

In [45]:
pip install tensorflow


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [46]:
import os
import urllib.request as request
from zipfile import ZipFile
from pathlib import Path
import tensorflow as tf

In [ ]:
class PrepareBaseModel:
    def __init__(self, config: PrepareBaseModelConfig):
        self.config = config

    def get_base_model(self):
        try:
            # FIX: Assign to self.model, not base_model
            self.model = tf.keras.applications.vgg16.VGG16(
                input_shape=self.config.params_image_size,
                include_top=self.config.params_include_top,
                weights=self.config.params_weights  
            )
            # Now self.model exists and can be saved
            self.save_model(path=self.config.base_model_path, model=self.model)
            
        except Exception as e:
            raise e 
    
    @staticmethod
    def prepare_full_model(model, classes, freeze_all, freeze_till, learning_rate):
        if freeze_all:
            for layer in model.layers:
                model.trainable = False # Freeze the whole base
        elif (freeze_till is not None) and (freeze_till > 0):
            for layer in model.layers[:-freeze_till]:
                layer.trainable = False

        # FIX: These layers must be OUTSIDE the elif block 
        # so they attach regardless of how you freeze.
        flatten_in = tf.keras.layers.Flatten()(model.output)
        prediction = tf.keras.layers.Dense(
            units=classes, 
            activation="softmax"
        )(flatten_in)

        full_model = tf.keras.Model(inputs=model.input, outputs=prediction)

        full_model.compile(
            optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
            loss=tf.keras.losses.CategoricalCrossentropy(),
            metrics=["accuracy"]
        )

        full_model.summary()
        return full_model
        
    def update_base_model(self):
        # This will now work because self.model was set in get_base_model
        self.full_model = self.prepare_full_model(
            model=self.model,
            classes=self.config.params_classes,
            freeze_all=True,
            freeze_till=None,
            learning_rate=self.config.params_learning_rate
        )

        self.save_model(path=self.config.updated_base_model_path, model=self.full_model)

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)

In [48]:
try:
    config = ConfiguarationManager()
    prepare_base_model_config = config.get_prepare_base_model_config()
    print(prepare_base_model_config)

    prepare_base_model = PrepareBaseModel(config=prepare_base_model_config)
    prepare_base_model.get_base_model()
    prepare_base_model.update_base_model()
except Exception as e:
    raise e

[2026-03-06 14:07:42,301: INFO: common]: yaml file: config/config.yaml loaded successfully
[2026-03-06 14:07:42,305: INFO: common]: yaml file: params.yaml loaded successfully
[2026-03-06 14:07:42,306: INFO: common]: created directory at: artifacts
[2026-03-06 14:07:42,308: INFO: common]: created directory at: artifacts/prepare_base_model
Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/home/savio-sunny/Desktop/testlearn/Kidney-Tumor-Classification-Project-using-MLflow-and-DVC-and-AWS/mlvenv/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 3747, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_92032/1125838345.py", line 10, in <module>
    raise e
  File "/tmp/ipykernel_92032/1125838345.py", line 3, in <module>
    prepare_base_model_config = config.get_prepare_base_model_config()
                                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_92032/37567295.py", line 14, in get_prepare_base_model_config
    prepare_base_model_config = PrepareBaseModelConfig(
                                ^^^^^^^^^^^^^^^^^^^^^^^
TypeError: PrepareBaseModelConfig.__init__() got an unexpected keyword argument 'params_learning_rate'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/savi